<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/LLAMAdotCPP_TOPO_CBP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Sovereign Edge Audit: Deterministic Bare-Metal CPU Inference via llama.cpp
# Model: frankmorales2020/certified-scorpion-topo-cbp-qwen-master-GGUF
# Checkpoint: certified-scorpion-topo-qwen-bf16.gguf (14.19 GB)
# ==============================================================================

import os
import subprocess
import sys

# 1. Configuration & Constants
WORKDIR = "/content/scorpion_edge_audit"
REPO_ID = "frankmorales2020/certified-scorpion-topo-cbp-qwen-master-GGUF"
MODEL_FILE = "certified-scorpion-topo-qwen-bf16.gguf"
MODEL_URL = f"https://huggingface.co/{REPO_ID}/resolve/main/{MODEL_FILE}"
SEED = 123
THREADS = os.cpu_count() or 2

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

print(f"[*] Working Directory: {WORKDIR}")
print(f"[*] Detected CPU Threads: {THREADS}")

# 2. Build llama.cpp for Native CPU Execution
if not os.path.exists("llama.cpp"):
    print("\n[*] Cloning llama.cpp repository...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git"], check=True)

print("\n[*] Configuring and compiling llama.cpp with native CPU SIMD optimization...")
cmake_configure_cmd = [
    "cmake", "-B", "build",
    "-DCMAKE_BUILD_TYPE=Release",
    "-DGGML_NATIVE=ON",
    "-DGGML_AVX=ON",
    "-DGGML_AVX2=ON",
    "-DGGML_F16C=ON",
    "-DGGML_FMA=ON",
    "-DGGML_OPENMP=ON",
    "-DGGML_CUDA=OFF",
    "-DGGML_METAL=OFF"
]
subprocess.run(cmake_configure_cmd, cwd=os.path.join(WORKDIR, "llama.cpp"), check=True)

cmake_build_cmd = [
    "cmake", "--build", "build",
    "--config", "Release",
    "-j", str(THREADS),
    "--target", "llama-cli"
]
subprocess.run(cmake_build_cmd, cwd=os.path.join(WORKDIR, "llama.cpp"), check=True)

# 3. Retrieve GGUF Binary from Hugging Face
model_path = os.path.join(WORKDIR, MODEL_FILE)
if not os.path.exists(model_path):
    print(f"\n[*] Downloading {MODEL_FILE} (14.19 GB)...")
    curl_cmd = ["curl", "-f", "-L", "--retry", "5", "-C", "-", MODEL_URL, "-o", MODEL_FILE]
    subprocess.run(curl_cmd, check=True)
else:
    file_size_gb = os.path.getsize(model_path) / (1024 ** 3)
    print(f"\n[*] Model binary present: {file_size_gb:.2f} GB")

# 4. Verify GGUF Header Integrity
with open(model_path, "rb") as f:
    magic = f.read(4)
    if magic != b"GGUF":
        raise ValueError(f"Integrity check failed: Expected b'GGUF', found {magic}")
print("[*] GGUF binary magic header confirmed.")

# 5. Locate Compiled Binary
bin_candidates = [
    os.path.join(WORKDIR, "llama.cpp", "build", "bin", "llama-cli"),
    os.path.join(WORKDIR, "llama.cpp", "build", "llama-cli")
]
llama_cli_bin = next((p for p in bin_candidates if os.path.isfile(p)), None)

if not llama_cli_bin:
    raise FileNotFoundError("Could not locate compiled llama-cli binary in build directory.")

# 6. Execute Deterministic Screening Prompt on CPU
prompt = (
    "<|im_start|>system\n"
    "You are a high-precision biophysical sequence screening engine. "
    "Evaluate the candidate construct for active binding against AaHII scorpion neurotoxin.<|im_end|>\n"
    "<|im_start|>user\n"
    "Analyze candidate construct: VRK-23.<|im_end|>\n"
    "<|im_start|>assistant\n"
)

# Cleaned arguments: standard llama-cli completion flags only
run_cmd = [
    llama_cli_bin,
    "-m", model_path,
    "-t", str(THREADS),
    "-s", str(SEED),
    "--temp", "0.0",
    "--top-p", "1.0",
    "-c", "1024",
    "-n", "128",
    "-r", "<|im_end|>",
    "-p", prompt
]

print("\n[*] Launching deterministic CPU inference (seed 123, temp 0.0)...")
# Passing input="" closes stdin immediately so llama-cli does not halt on an interactive prompt
res = subprocess.run(run_cmd, input="", capture_output=True, text=True)

if res.returncode != 0:
    print(f"\n[!] Inference execution failed with exit code: {res.returncode}")
    print("[!] llama.cpp stderr trace:\n")
    print(res.stderr)
    sys.exit(res.returncode)

print("\n--- Output ---")
print(res.stdout)
print("\n[*] Audit complete.")

[*] Working Directory: /content/scorpion_edge_audit
[*] Detected CPU Threads: 2

[*] Cloning llama.cpp repository...

[*] Configuring and compiling llama.cpp with native CPU SIMD optimization...

[*] Downloading certified-scorpion-topo-qwen-bf16.gguf (14.19 GB)...
[*] GGUF binary magic header confirmed.

[*] Launching deterministic CPU inference (seed 123, temp 0.0)...
